# Generate well-level bulk profiles

NOTE: We are normalizing the bulk profiles to the vehicle-treated wells (negative controls) as the "standard".

## Import libraries

In [1]:
import pathlib

import pandas as pd
from pycytominer import aggregate, annotate, feature_select, normalize
from pycytominer.cyto_utils import output


## Set paths and variables

In [2]:
# Set the plate to process
plate_id = "CARD-CelIns-CX7_261022150001"

# Directory with QC-labeled profiles
qc_labeled_dir = pathlib.Path("./data/qc_labeled_profiles/").resolve(strict=True)

# Directory to save bulk profiles
output_dir = pathlib.Path("./data/bulk_profiles/")
output_dir.mkdir(parents=True, exist_ok=True)

# Path to the platemap for the pilot plate
platemap_path = pathlib.Path(
    "../metadata/platemaps/pilot_platemap.csv"
).resolve(strict=True)

# Path to the QC-labeled profile for the pilot plate
profile_path = (qc_labeled_dir / f"{plate_id}_qc_labeled.parquet").resolve(
    strict=True
)

# operations to perform for feature selection
# frequency_threshold removes features dominated by one value (in newer pycytominer versions, e.g., 1.7.1,
# variance_threshold only removes features with low raw variance)
feature_select_ops = [
    "blocklist",
    "drop_na_columns",
    "variance_threshold",
    "frequency_threshold",
    "correlation_threshold",
]


## Process data with pycytominer


In [3]:
print("Performing preprocessing on", plate_id)

# generating all output file paths
output_annotated_file = str(output_dir / f"{plate_id}_bulk_annotated.parquet")
output_normalized_file = str(output_dir / f"{plate_id}_bulk_normalized.parquet")
output_feature_select_file = str(
    output_dir / f"{plate_id}_bulk_feature_selected.parquet"
)
output_spherized_file = str(output_dir / f"{plate_id}_bulk_spherized.parquet")

# loading profiles
profile_df = pd.read_parquet(profile_path)
platemap_df = pd.read_csv(platemap_path)

# Drop all rows in the profiles that failed any Metadata_cqc columns
cqc_columns = [col for col in profile_df.columns if col.startswith("Metadata_cqc")]
if cqc_columns:
    profile_df = profile_df[~profile_df[cqc_columns].any(axis=1)]

# Step 1: Aggregate single-cell data to the well-level using the median
print("Performing aggregation for", plate_id, "...")
aggregated_df = aggregate(
    population_df=profile_df,
    operation="median",
    strata=["Image_Metadata_Plate", "Image_Metadata_Well"],
)

# Step 2: Annotation
print("Performing annotation for", plate_id, "...")
annotated_df = annotate(
    profiles=aggregated_df,
    platemap=platemap_df,
    join_on=["Metadata_well_position", "Image_Metadata_Well"],
)

# Fix metadata column names
annotated_df = annotated_df.rename(columns={"Image_Metadata_Site": "Metadata_Site"})

# Save annotated profiles
output(
    df=annotated_df,
    output_filename=output_annotated_file,
    output_type="parquet",
)

# Step 3: Normalization (mad robustize)
# Normalize using the negative controls as the reference population
print("Performing normalization for", plate_id, "...")
neg_control_query = "Metadata_Treatment == 'Veh'"
normalize(
    profiles=annotated_df,
    method="mad_robustize",
    samples=neg_control_query,
    output_type="parquet",
    output_file=output_normalized_file,
)

# Step 4: Feature selection
print("Performing feature selection for", plate_id, "...")
feature_select_df = feature_select(
    profiles=output_normalized_file,
    operation=feature_select_ops,
    na_cutoff=0,
    blocklist_file="./blocklist_features.txt",
    corr_threshold=0.95,
    freq_cut=0.05,
    output_type="parquet",
    output_file=output_feature_select_file,
)

# Step 4b: Remove features with too little variation inside the exact control
# population used to fit spherization.
print(
    f"Feature selecting {plate_id} with variance threshold "
    "within negative controls only..."
)
zero_negcon_var_fs_df = feature_select(
    profiles=feature_select_df,
    operation=["variance_threshold", "frequency_threshold"],
    freq_cut=0.05,
    unique_cut=0.01,
    samples=neg_control_query,
)

# Step 5: Spherize/whiten all profiles using the negative controls as the
# reference population.
print(f"Sphering {plate_id} using negative controls...")
normalize(
    profiles=zero_negcon_var_fs_df,
    method="spherize",
    samples=neg_control_query,
    spherize_center=True,
    spherize_method="ZCA-cor",
    spherize_epsilon=1e-6,
    output_file=output_spherized_file,
    output_type="parquet",
)

print(f"Saved feature-selected profiles to {output_feature_select_file}")
print(f"Saved spherized profiles to {output_spherized_file}")


Performing preprocessing on CARD-CelIns-CX7_261022150001


Performing aggregation for CARD-CelIns-CX7_261022150001 ...


Performing annotation for CARD-CelIns-CX7_261022150001 ...


Performing normalization for CARD-CelIns-CX7_261022150001 ...


Performing feature selection for CARD-CelIns-CX7_261022150001 ...


Feature selecting CARD-CelIns-CX7_261022150001 with variance threshold within negative controls only...


Sphering CARD-CelIns-CX7_261022150001 using negative controls...


Saved feature-selected profiles to data/bulk_profiles/CARD-CelIns-CX7_261022150001_bulk_feature_selected.parquet
Saved spherized profiles to data/bulk_profiles/CARD-CelIns-CX7_261022150001_bulk_spherized.parquet
